# Feature Engineering

---

1. Import packages
2. Load data
3. Create price features
4. Create date features
5. Merge datasets
6. Final feature set

---

## 1. Import packages

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Data

In [2]:
# Load cleaned data
df = pd.read_csv('clean_data_after_eda.csv')

print(f"Data loaded successfully!")
print(f"Shape: {df.shape[0]:,} rows | {df.shape[1]} columns")
display(df.head())

Data loaded successfully!
Shape: 14,606 rows | 44 columns


,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,0.000011,2.896760e-06,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0


In [5]:
# Load price data as well
price_df = pd.read_csv('price_data (1).csv')
price_df['price_date'] = pd.to_datetime(price_df['price_date'])

print(f"Price data loaded!")
print(f"Shape: {price_df.shape[0]:,} rows | {price_df.shape[1]} columns")

Price data loaded!
Shape: 193,002 rows | 8 columns


## 3. Create Price Features

In [6]:
# Parse price_date column
price_df['price_date'] = pd.to_datetime(price_df['price_date'])

# Extract month and year
price_df['month'] = price_df['price_date'].dt.month
price_df['year']  = price_df['price_date'].dt.year

# Estelle's Feature — Difference between off-peak prices
# in December and January of the preceding year
# This captures how much prices changed year-over-year
# which could signal price sensitivity

dec_prices = price_df[price_df['month'] == 12].groupby('id')[
    ['price_off_peak_var', 'price_off_peak_fix']].mean().reset_index()
dec_prices.columns = ['id', 'dec_price_off_peak_var', 'dec_price_off_peak_fix']

jan_prices = price_df[price_df['month'] == 1].groupby('id')[
    ['price_off_peak_var', 'price_off_peak_fix']].mean().reset_index()
jan_prices.columns = ['id', 'jan_price_off_peak_var', 'jan_price_off_peak_fix']

# Merge December and January prices
price_diff = dec_prices.merge(jan_prices, on='id', how='left')

# Calculate the difference — Estelle's suggested feature
price_diff['price_diff_off_peak_var'] = (
    price_diff['dec_price_off_peak_var'] - price_diff['jan_price_off_peak_var']
)
price_diff['price_diff_off_peak_fix'] = (
    price_diff['dec_price_off_peak_fix'] - price_diff['jan_price_off_peak_fix']
)

print("Estelle's price difference features created!")
print(f"\nFeatures created:")
print(f"  price_diff_off_peak_var : Dec vs Jan off-peak variable price difference")
print(f"  price_diff_off_peak_fix : Dec vs Jan off-peak fixed price difference")
print(f"\nSample values:")
display(price_diff[['id', 'price_diff_off_peak_var', 'price_diff_off_peak_fix']].head())
print(f"\nStatistics:")
display(price_diff[['price_diff_off_peak_var', 'price_diff_off_peak_fix']].describe().round(4))

Estelle's price difference features created!

Features created:
  price_diff_off_peak_var : Dec vs Jan off-peak variable price difference
  price_diff_off_peak_fix : Dec vs Jan off-peak fixed price difference

Sample values:


,id,price_diff_off_peak_var,price_diff_off_peak_fix
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001



Statistics:


,price_diff_off_peak_var,price_diff_off_peak_fix
count,16068.0000,16068.0000
mean,-0.0045,0.2778
std,0.0128,1.4313
min,-0.1485,-44.2669
25%,-0.0082,0.0000
50%,-0.0056,0.1629
75%,-0.0036,0.1778
max,0.1690,40.7289


### Observation — Estelle's Price Feature

- The mean price difference (off-peak var) is **-0.0045** — meaning 
  on average, December prices are slightly **lower** than January
- Most customers see very small price changes year-over-year
- However, some outliers exist — max difference of **0.169** — 
  these customers experienced significant price hikes
- **Hypothesis:** Customers who experienced larger price increases 
  in December vs January may be more likely to churn

## 4. Create Date Features

In [7]:
# Convert date columns to datetime
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Reference date
ref_date = pd.Timestamp('2016-01-01')

# Days since activation — older customers more stable
df['days_activ']        = (ref_date - df['date_activ']).dt.days

# Days until contract end — customers near end more likely to churn
df['days_to_end']       = (df['date_end'] - ref_date).dt.days

# Days since last product modification
df['days_modif_prod']   = (ref_date - df['date_modif_prod']).dt.days

# Days until renewal — closer to renewal = higher churn risk
df['days_to_renewal']   = (df['date_renewal'] - ref_date).dt.days

print("Date features created!")
print(f"\nNew features:")
print(f"  days_activ      : Days since contract activation")
print(f"  days_to_end     : Days until contract ends")
print(f"  days_modif_prod : Days since last product modification")
print(f"  days_to_renewal : Days until next renewal")
display(df[['days_activ', 'days_to_end',
            'days_modif_prod', 'days_to_renewal']].describe().round(2))

Date features created!

New features:
  days_activ      : Days since contract activation
  days_to_end     : Days until contract ends
  days_modif_prod : Days since last product modification
  days_to_renewal : Days until next renewal


,days_activ,days_to_end,days_modif_prod,days_to_renewal
count,14606.00,14606.00,14606.00,14606.00
mean,1798.67,208.87,1093.48,-163.71
std,589.32,106.93,922.47,118.41
min,487.00,27.00,-28.00,-919.00
25%,1352.00,117.25,199.00,-259.00
50%,1764.00,213.00,926.00,-158.00
75%,2177.00,304.00,1968.00,-64.00
max,4620.00,529.00,4620.00,27.00


### Observation — Date Features

- Average contract activation was **~1,798 days ago (~5 years)** — 
  PowerCo has a mature customer base
- `days_to_renewal` has **negative values** — many customers are 
  already past their renewal date but still active
- Customers close to contract end (`days_to_end` low) may be 
  evaluating competitors — **high churn risk window**
- `days_modif_prod` — customers who haven't modified their product 
  in a long time may be disengaged — another churn signal

## 5. Merge Datasets

In [8]:
# Merge price difference features with main dataframe
df_final = df.merge(price_diff[['id',
                                 'price_diff_off_peak_var',
                                 'price_diff_off_peak_fix']],
                    on='id', how='left')

# Drop original date columns — already extracted useful features
df_final = df_final.drop(columns=['date_activ', 'date_end',
                                   'date_modif_prod', 'date_renewal'])

print("Datasets merged successfully!")
print(f"\nFinal dataset shape: {df_final.shape[0]:,} rows | {df_final.shape[1]} columns")
print(f"New columns added   : price_diff_off_peak_var, price_diff_off_peak_fix")
print(f"Columns dropped     : date_activ, date_end, date_modif_prod, date_renewal")
print(f"\nSample of new features:")
display(df_final[['id', 'price_diff_off_peak_var',
                   'price_diff_off_peak_fix',
                   'days_activ', 'days_to_end',
                   'days_to_renewal', 'churn']].head())

Datasets merged successfully!

Final dataset shape: 14,606 rows | 46 columns
New columns added   : price_diff_off_peak_var, price_diff_off_peak_fix
Columns dropped     : date_activ, date_end, date_modif_prod, date_renewal

Sample of new features:


,id,price_diff_off_peak_var,price_diff_off_peak_fix,days_activ,days_to_end,days_to_renewal,churn
0,24011ae4ebbe3035111d65fa7c15bc57,0.020057,3.700961,930,166,-192,1
1,d29c2c54acc38ff3c0614d0a653813dd,-0.003767,0.177779,2324,242,-123,0
2,764c75f661154dac3a6c254cd082ea7d,-0.004670,0.177779,2086,106,-259,0
3,bba03439a292a1e166f80264c16191cb,-0.004547,0.177779,2103,89,-276,0
4,149d57cf92fc41cf94415803a877cb4b,-0.006192,0.162916,2179,66,-298,0


### Observation — Merged Dataset

- Successfully merged price features with customer data on `id`
- Date columns dropped — replaced with more meaningful numeric 
  features (days_activ, days_to_end etc.)
- Final dataset now has **46 columns** — richer feature set 
  compared to original 26 columns

## 6. Final Feature Set

In [9]:
# Drop irrelevant columns
# id — just identifier, not predictive
# activity_new, channel_sales, origin_up — hashed strings, 
# already encoded in cleaned data

cols_to_drop = ['id', 'activity_new', 'channel_sales', 'origin_up']
df_final = df_final.drop(columns=cols_to_drop, errors='ignore')

# Encode has_gas — binary categorical
df_final['has_gas'] = (df_final['has_gas'] == 't').astype(int)

# Check for remaining missing values
missing = df_final.isnull().sum()
missing = missing[missing > 0]

print("Final Feature Set Ready!")
print(f"\nFinal shape : {df_final.shape[0]:,} rows | {df_final.shape[1]} columns")
print(f"Target var  : churn — {df_final['churn'].sum():,} churned ({df_final['churn'].mean()*100:.1f}%)")

if len(missing) > 0:
    print(f"\nMissing values found:")
    print(missing)
else:
    print(f"\nNo missing values — clean dataset ready for modeling!")

# Save final dataset
df_final.to_csv('final_features.csv', index=False)
print(f"\nSaved: final_features.csv")
print(f"Total features for modeling: {df_final.shape[1]-1}")

Final Feature Set Ready!

Final shape : 14,606 rows | 43 columns
Target var  : churn — 1,419 churned (9.7%)

Missing values found:
price_diff_off_peak_var    22
price_diff_off_peak_fix    22
dtype: int64

Saved: final_features.csv
Total features for modeling: 42


## 7. Handle Missing Values

In [10]:
# Fill missing price diff values with median
# Median is more robust than mean for price data
df_final['price_diff_off_peak_var'] = df_final['price_diff_off_peak_var'].fillna(
    df_final['price_diff_off_peak_var'].median()
)
df_final['price_diff_off_peak_fix'] = df_final['price_diff_off_peak_fix'].fillna(
    df_final['price_diff_off_peak_fix'].median()
)

# Verify no missing values remain
missing_check = df_final.isnull().sum().sum()

print("Missing values handled!")
print(f"\nMissing values remaining : {missing_check}")
print(f"Final shape              : {df_final.shape[0]:,} rows | {df_final.shape[1]} columns")
print(f"Total features for model : {df_final.shape[1]-1}")

# Save final clean dataset
df_final.to_csv('final_features.csv', index=False)
print(f"\nFinal dataset saved: final_features.csv")
print(f"\nFeature Engineering Complete!")
print(f"Dataset is ready for Random Forest modeling!")

Missing values handled!

Missing values remaining : 0
Final shape              : 14,606 rows | 43 columns
Total features for model : 42

Final dataset saved: final_features.csv

Feature Engineering Complete!
Dataset is ready for Random Forest modeling!


### Final Summary — Feature Engineering

**Features Created:**
- `price_diff_off_peak_var` — Dec vs Jan variable price change (Estelle's idea)
- `price_diff_off_peak_fix` — Dec vs Jan fixed price change
- `days_activ` — Customer tenure in days
- `days_to_end` — Days until contract expires
- `days_modif_prod` — Days since last product change
- `days_to_renewal` — Days until/since renewal date

**Features Removed:**
- `id`, `activity_new`, `channel_sales`, `origin_up` — identifiers/hashed strings
- Date columns — replaced with more meaningful numeric features

**Why these features matter:**
- Price difference captures **year-over-year price sensitivity**
- Days features capture **contract lifecycle stage** — 
  a strong behavioral signal for churn
- `has_gas` encoded as binary — ready for ML model

**Dataset is now ready for Random Forest Modeling!**
- 14,606 customers | 42 features | 1,419 churned (9.7%)